# Notebook 02 — SAE Feature Analysis

Load a trained TFIMTransformer + TopK SAE and analyse the discovered features:
- Dead feature fraction over training
- Activation density histograms
- Top-activating inputs per feature
- Feature-to-observable Pearson correlation heatmap

**Prerequisites:** run `exp_ra02_observables.py` first to generate `runs/ra02_observables/`.

In [ ]:
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

RUN_DIR = Path('../runs/ra02_observables')

In [ ]:
# --- Load saved activations and observables ---
data = torch.load(RUN_DIR / 'activations.pt', weights_only=False)

acts     = data['activations'].numpy()    # (N, d_model)
z_sae    = data['z_sae'].numpy()          # (N, d_hidden)
r_mat    = data['r_mat']                  # (F_alive, O)
obs_names = data['obs_names']

print(f'Activations: {acts.shape}')
print(f'SAE features: {z_sae.shape}')
print(f'Observables: {obs_names}')

In [ ]:
# --- SAE summary ---
with open(RUN_DIR / 'sae_summary.json') as f:
    sae_summary = json.load(f)
print('SAE summary:')
for k, v in sae_summary.items():
    print(f'  {k}: {v}')

In [ ]:
# --- Correlation heatmap: top-30 features ---
max_r = np.abs(r_mat).max(axis=1)
top30 = np.argsort(-max_r)[:30]

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(r_mat[top30].T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(top30)))
ax.set_xticklabels([f'f{i}' for i in top30], rotation=90, fontsize=8)
ax.set_yticks(range(len(obs_names)))
ax.set_yticklabels(obs_names)
ax.set_title('Pearson r: SAE features × quantum observables (top 30 by max |r|)')
plt.colorbar(im, ax=ax, label='Pearson r')
plt.tight_layout()
plt.show()

In [ ]:
# --- Top features per observable ---
with open(RUN_DIR / 'top_features.json') as f:
    top_feats = json.load(f)

for obs_name, entries in top_feats.items():
    best = entries[0]
    print(f'{obs_name:20s}  best: feat={best["feature_idx"]:3d}  '
          f'r={best["pearson_r"]:+.4f}  p={best["p_value"]:.2e}')

In [ ]:
# --- Activation density per feature (alive features only) ---
alive_mask = (z_sae > 0).any(axis=0)
z_alive    = z_sae[:, alive_mask]
density    = (z_alive > 0).mean(axis=0)

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(density, bins=30, color='steelblue', edgecolor='white')
ax.set_xlabel('Activation density (fraction of inputs where feature fires)')
ax.set_ylabel('Count')
ax.set_title(f'Feature density distribution ({alive_mask.sum()} alive features)')
plt.tight_layout()
plt.show()